# Лабораторна робота № 4
## Дослідження дерев рішень та ансамблевих методів у задачі багатокласової класифікації за наявності пропущених значень

**Варіант:** N = 4, **SEED = 42 + N = 46** (єдина константа для поділу даних, крос-валідації, моделей і семплера Optuna).

**Датасет:** Annealing (OpenML, `name='anneal', version=1`).

> ⏱ **Час виконання.** Згідно з методичкою `N_TRIALS = 30`. Вкладена крос-валідація виконує 3 моделі × 5 зовнішніх блоків × 30 спроб × 3 внутрішніх блоки, тому повний запуск може тривати десятки хвилин. Для швидкої перевірки працездатності тимчасово зменште `N_TRIALS`, але **для звіту поверніть 30**.
>
> Усі числа в таблицях обчислюються під час запуску (Restart & Run All). Markdown-клітинки з позначкою ✍️ призначені для власних висновків за вашими числами.

In [ ]:
# Встановлення бібліотек (потрібно в Google Colab). Після виконання, якщо Colab запропонує,
# перезапустіть середовище (Runtime → Restart session) і запускайте ноутбук далі з наступної клітинки.
%pip install -q optuna shap xgboost openml

In [ ]:
import os, sys, time, random, warnings, subprocess, platform
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*is experimental.*")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*Skipping features without any observed values.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
import shap
import sklearn, xgboost, scipy
from IPython.display import display

from scipy.stats import t as t_dist, chi2_contingency, spearmanr
from sklearn.datasets import fetch_openml
from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_validate, cross_val_score, cross_val_predict)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.inspection import permutation_importance

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ── Індивідуальний варіант ──
N = 4
SEED = 42 + N
random.seed(SEED); np.random.seed(SEED)

N_TRIALS = 30            # за методичкою; для швидкої перевірки можна тимчасово зменшити
MODELS = ["DT", "RF", "XGB"]
MODEL_LABELS = {"DT": "Decision Tree", "RF": "Random Forest", "XGB": "XGBoost"}
SCORING = {"acc": "accuracy", "f1m": "f1_macro", "bacc": "balanced_accuracy"}
os.makedirs("models", exist_ok=True)

# ── Версії бібліотек (вимога методички) + requirements.txt ──
print("Python", platform.python_version())
for lib in (sklearn, xgboost, optuna, shap, pd, np, scipy):
    print(f"{lib.__name__:<10} {lib.__version__}")
with open("requirements.txt", "w") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=False)
print("\nSEED =", SEED, "| N_TRIALS =", N_TRIALS, "| requirements.txt збережено")

## 3.1 Первинний аналіз даних

In [ ]:
data = fetch_openml(name="anneal", version=1, as_frame=True)
X, y = data.data.copy(), data.target.copy()

# Категоріальні ознаки -> object (пропуски лишаються NaN)
for col in X.columns:
    if not pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].astype("object")
y_str = y.astype(str)

n_num = sum(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns)
print("Версія датасету: OpenML anneal v1")
print(f"Спостережень: {X.shape[0]}, ознак: {X.shape[1]} (числових: {n_num}, категоріальних/object: {X.shape[1]-n_num})")
print("Кількість класів у даних:", y_str.nunique())

miss_table = pd.DataFrame({
    "Тип": X.dtypes.astype(str),
    "Пропущено": X.isna().sum(),
    "Частка пропусків": X.isna().mean().round(3),
    "Унікальних (без NaN)": X.nunique(dropna=True),
}).sort_values("Частка пропусків", ascending=False)
display(miss_table)

### Графік 2 — частка пропусків за ознаками

In [ ]:
share = X.isna().mean().sort_values(ascending=False)
share = share[share > 0]
fig, ax = plt.subplots(figsize=(12, 4.5))
share.plot(kind="bar", ax=ax, color=["tab:red" if v == 1 else "tab:blue" for v in share])
ax.set_title("Частка пропущених значень за ознаками (червоні — повністю порожні)")
ax.set_ylabel("Частка пропусків"); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

### Видалення порожніх і константних ознак

* **Повністю порожні** ознаки (100 % `NaN`) не містять жодної інформації — видаляються.
* **Константні** ознаки (одне й те саме значення в усіх рядках, **без** пропусків) не можуть розділяти класи — видаляються.
* Ознаки, що мають **одне значення + пропуски** (наприклад, «Y» або `NaN`), **не** вважаються константними: сама наявність/відсутність значення тут і є інформацією (це саме те, що ловлять індикатори пропусків у варіанті B). Тому вони залишаються.

In [ ]:
all_missing = X.columns[X.isna().all()].tolist()
constant = [c for c in X.columns if c not in all_missing and X[c].nunique(dropna=False) <= 1]
one_value_plus_nan = [c for c in X.columns
                      if c not in all_missing and X[c].nunique(dropna=True) == 1 and X[c].isna().any()]
print("Повністю порожні:", all_missing)
print("Константні (без пропусків):", constant)
print("Одне значення + пропуски (ЗАЛИШЕНО, несуть інформацію через пропуск):", one_value_plus_nan)

X = X.drop(columns=all_missing + constant)
print("\nПісля видалення:", X.shape)

### Графік 1 — розподіл класів цільової змінної та рішення щодо рідкісних класів

In [ ]:
class_counts = y_str.value_counts().sort_index()
display(pd.DataFrame({"Кількість": class_counts, "Частка": (class_counts / len(y_str)).round(4)}))

fig, ax = plt.subplots(figsize=(7, 4))
class_counts.plot(kind="bar", ax=ax, color="tab:blue")
for i, v in enumerate(class_counts.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_title("Розподіл класів цільової змінної")
ax.set_xlabel("Клас"); ax.set_ylabel("Кількість")
plt.tight_layout(); plt.show()

print("Найменший клас:", class_counts.idxmin(), "—", class_counts.min(), "спостережень.")
print("Клас, описаний у документації, але відсутній у даних, не потрапляє в LabelEncoder.")

**Рішення щодо рідкісних класів.** Найменший клас містить лише кілька спостережень. Якщо його видалити або об'єднати з іншим, ми змінимо задачу й втратимо клас, який саме і цікавий. Тому класи **залишаємо**, але обов'язково перевіряємо кількість об'єктів у *тренувальній* частині після поділу (дивись п. 3.2): для стратифікованої CV з 5 блоками кожен клас повинен мати ≥ 5 тренувальних об'єктів; тоді він присутній у кожному блоці, а внутрішня 3-блочна CV всередині зовнішнього тренувального блоку теж працює без попереджень і без помилки XGBoost через «зниклий» клас.

### Аналіз механізму пропусків (топ-5 ознак за часткою пропусків)

Для кожної ознаки будуємо таблицю спряженості «пропуск / є значення» × клас і перевіряємо незалежність за χ². Якщо очікувані частоти малі (< 5), асимптотичний χ² ненадійний (а точний критерій Фішера в SciPy доступний лише для 2×2), тому p-значення оцінюємо **перестановочним (Monte-Carlo) χ²-тестом** — це аналог точного критерію для таблиць r×c.

> Це розвідувальний аналіз усього датасету; він не впливає на навчання моделей.

In [ ]:
def chi2_stat(tab):
    exp = tab.sum(1, keepdims=True) * tab.sum(0, keepdims=True) / tab.sum()
    m = exp > 0
    return (((tab - exp) ** 2)[m] / exp[m]).sum()

def mc_chi2_pvalue(miss_flag, labels, n_perm=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    K = labels.max() + 1
    def table(lbl):
        return np.bincount(miss_flag * K + lbl, minlength=2 * K).reshape(2, K).astype(float)
    obs = chi2_stat(table(labels))
    cnt = sum(chi2_stat(table(rng.permutation(labels))) >= obs - 1e-12 for _ in range(n_perm))
    return (cnt + 1) / (n_perm + 1)

top5 = X.isna().mean().sort_values(ascending=False)
top5 = top5[(top5 > 0) & (top5 < 1)].head(5).index.tolist()
lab_codes = pd.factorize(y_str)[0]

chi_rows = []
for col in top5:
    flag = X[col].isna().values.astype(int)
    tab = pd.crosstab(pd.Series(np.where(flag == 1, "пропуск", "є значення"), name=col), y_str.rename("клас"))
    print(f"\n── {col} (частка пропусків = {flag.mean():.3f}) ──")
    display(tab)
    chi2, p_asym, dof, expected = chi2_contingency(tab.values)
    small = (expected < 5).any()
    p_final = mc_chi2_pvalue(flag, lab_codes) if small else p_asym
    cramers_v = np.sqrt(chi2 / (tab.values.sum() * (min(tab.shape) - 1)))
    chi_rows.append({"Ознака": col, "Частка пропусків": round(flag.mean(), 3), "χ²": round(chi2, 2), "df": dof,
                     "Є очікувані < 5": small, "Метод p-значення": "Monte-Carlo χ²" if small else "асимптотичний χ²",
                     "p-значення": p_final, "Cramér's V": round(cramers_v, 3),
                     "Висновок": "пропуск пов'язаний з класом (не MCAR)" if p_final < 0.05 else "немає доказів проти MCAR"})
chi_df = pd.DataFrame(chi_rows)
display(chi_df)
n_dep = int((chi_df["p-значення"] < 0.05).sum())
print(f"\nУ {n_dep} із {len(chi_df)} ознак пропуск статистично пов'язаний з класом (α = 0,05).")

✍️ **Коментар до механізму пропусків.** Якщо p-значення малі, пропуск несе інформацію про клас — гіпотеза MCAR відхиляється, а наявність пропуску сама є ознакою (звідси потенційна користь індикаторів у варіанті B). Критерій не розрізняє MAR та MNAR: для цього потрібні знання про фізику процесу (в Annealing частина пропусків *структурна* — ознака просто не застосовна до об'єкта). Запишіть тут власний висновок за вашою таблицею.

## 3.1 (п. 6) Кодування цілі та 3.2 Поділ даних

In [ ]:
le = LabelEncoder()
y_enc = pd.Series(le.fit_transform(y_str), index=y.index, name="target")
class_names = list(le.classes_)
print("Кодування класів:", dict(zip(class_names, range(len(class_names)))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=SEED
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

cnt = pd.DataFrame({"train": y_train.value_counts().sort_index(), "test": y_test.value_counts().sort_index()})
cnt.index = [class_names[i] for i in cnt.index]
display(cnt)

min_train = int(y_train.value_counts().min())
print("Мінімум об'єктів класу в train:", min_train)
if min_train < 5:
    raise ValueError("Клас має < 5 тренувальних об'єктів: 5-блочна CV неможлива — потрібно обґрунтовано об'єднати/виключити клас.")
print("Перевірка пройдена: кожен клас має ≥ 5 тренувальних об'єктів (5-блочна CV, внутрішня 3-блочна CV коректні).")
print("Об'єктів найменшого класу в test:", int(y_test.value_counts().min()),
      "— тестові метрики для цього класу будуть дуже нестабільними.")

## 3.2 Пайплайни обробки пропусків (варіанти A та B) та 3.3 Базові моделі

* **Варіант A:** числові — медіана; категоріальні — мода + one-hot.
* **Варіант B:** A + бінарні індикатори пропусків (`MissingIndicator(features='missing-only')`) для ознак, що мали пропуски **у тренувальних даних**.

Імпутація виконується **лише всередині `Pipeline`**, тож у кожному блоці CV вона навчається тільки на тренувальній частині блоку (немає витоку даних). Імпутацію застосовано навмисно (хоча DT/RF/XGB вміють працювати з `NaN`), щоб усі моделі отримували однакові вхідні дані.

In [ ]:
def build_preprocessor(with_indicators: bool) -> ColumnTransformer:
    transformers = [
        ("num", SimpleImputer(strategy="median"),
         make_column_selector(dtype_include="number")),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), make_column_selector(dtype_exclude="number")),
    ]
    if with_indicators:
        transformers.append(
            ("miss", MissingIndicator(features="missing-only"), make_column_selector()))
    return ColumnTransformer(transformers)

def make_model(name, params=None):
    params = params or {}
    if name == "DT":
        return DecisionTreeClassifier(random_state=SEED, **params)
    if name == "RF":
        return RandomForestClassifier(random_state=SEED, n_jobs=-1, **params)
    return XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="mlogloss", **params)

def build_pipeline(name, with_indicators=True, params=None):
    return Pipeline([("prep", build_preprocessor(with_indicators)),
                     ("clf", make_model(name, params))])

# Один і той самий об'єкт cv для всіх моделей і варіантів (потрібно для парного тесту в п. 3.5)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
_s1 = [te.tolist() for _, te in cv.split(X_train, y_train)]
_s2 = [te.tolist() for _, te in cv.split(X_train, y_train)]
assert _s1 == _s2, "Блоки CV мають бути однаковими при кожному виклику"
print("Блоків CV (k·r):", len(_s1), "— розбиття детерміноване (random_state = SEED).")

def evaluate_cv(with_indicators):
    raw = {}
    for name in MODELS:
        raw[name] = cross_validate(build_pipeline(name, with_indicators), X_train, y_train,
                                   cv=cv, scoring=SCORING, n_jobs=1)
    return raw

def format_cv_table(raw):
    rows = []
    for name in MODELS:
        r = {"Модель": MODEL_LABELS[name]}
        for key, label in [("test_acc", "Accuracy"), ("test_f1m", "Macro F1"), ("test_bacc", "Balanced acc.")]:
            v = raw[name][key]
            r[label] = f"{v.mean():.4f} ± {v.std(ddof=1):.4f}"
        r["Час fit, с (сер.)"] = round(raw[name]["fit_time"].mean(), 2)
        rows.append(r)
    return pd.DataFrame(rows).set_index("Модель")

### Таблиця 1 — крос-валідація (5×3), варіант A, типові параметри (середнє ± std)

In [ ]:
raw_A = evaluate_cv(with_indicators=False)
table1 = format_cv_table(raw_A)
display(table1)

### Таблиця 2 — крос-валідація (5×3), варіант B (з індикаторами), типові параметри (середнє ± std)

In [ ]:
raw_B = evaluate_cv(with_indicators=True)
table2 = format_cv_table(raw_B)
display(table2)

✍️ **Коментар до таблиць 1–2.** Через «ефект стелі» accuracy у дерев близька до 1, тому розбіжності між моделями малі; основна метрика — macro F1 (враховує рідкісні класи) разом зі стандартним відхиленням. Чи покращили індикатори пропусків macro F1 і чи перевищує різниця шум — перевіряється у п. 3.5.

## 3.4 Підбір гіперпараметрів за допомогою Optuna

У цільову функцію передається **весь пайплайн** (імпутація + модель), тож на кожному внутрішньому блоці імпутер навчається лише на тренувальній частині блоку. Внутрішня CV має 3 блоки (через рідкісні класи). Тестова вибірка в оптимізації **не використовується**.

In [ ]:
def suggest_params(trial, name):
    if name == "DT":
        return {
            "max_depth": trial.suggest_int("max_depth", 2, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        }
    if name == "RF":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
            "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
        }
    return {  # XGB
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10, log=True),
    }

def tune(name, X_tr, y_tr, n_trials=N_TRIALS):
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    def objective(trial):
        pipe = build_pipeline(name, True, suggest_params(trial, name))
        return cross_val_score(pipe, X_tr, y_tr, cv=inner, scoring="f1_macro").mean()
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    return study

### Оптимізація на всій тренувальній вибірці (для фінальних моделей)

In [ ]:
studies, tune_time = {}, {}
for name in MODELS:
    t0 = time.time()
    studies[name] = tune(name, X_train, y_train)
    tune_time[name] = time.time() - t0
    print(f"{MODEL_LABELS[name]:<14} best_value (macro F1, 3-fold CV) = {studies[name].best_value:.4f}  [{tune_time[name]:.0f} с]")

### Таблиця 4 — найкращі гіперпараметри, знайдені Optuna

In [ ]:
def _fmt(v): return f"{v:.4g}" if isinstance(v, float) else str(v)
all_params = sorted(set().union(*[studies[n].best_params.keys() for n in MODELS]))
table4 = pd.DataFrame({MODEL_LABELS[n]: [_fmt(studies[n].best_params[p]) if p in studies[n].best_params else "—"
                                         for p in all_params] for n in MODELS}, index=all_params)
table4.loc["best_value (macro F1)"] = [f"{studies[n].best_value:.4f}" for n in MODELS]
display(table4)

### Графік 3 — історія оптимізації та важливість гіперпараметрів

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

for name in MODELS:
    study = studies[name]
    ax = plot_optimization_history(study)
    ax.set_title(f"{MODEL_LABELS[name]}: історія оптимізації (macro F1)")
    plt.tight_layout(); plt.show()

    try:
        ax = plot_param_importances(study)
        ax.set_title(f"{MODEL_LABELS[name]}: важливість гіперпараметрів")
        plt.tight_layout(); plt.show()
        imps = optuna.importance.get_param_importances(study)
        print("Найважливіші параметри:", {k: round(v, 3) for k, v in list(imps.items())[:3]})
    except Exception as e:
        print("Важливість параметрів недоступна:", repr(e))

    vals = np.array([t.value for t in study.trials])
    best_so_far = np.maximum.accumulate(vals)
    first_best = int(np.argmax(best_so_far == best_so_far[-1]))
    print(f"{MODEL_LABELS[name]}: найкраще значення {best_so_far[-1]:.4f} досягнуто на спробі #{first_best}"
          f" із {len(vals)}; після 10 спроб було {best_so_far[min(9, len(vals)-1)]:.4f}.\n")

✍️ **Коментар до оптимізації.** Оцініть, чи «вийшла на плато» крива best-so-far (якщо найкраще значення знайдено в кінці — варто збільшити `N_TRIALS`), і які параметри мають найбільший вплив. Через ефект стелі багато конфігурацій дають майже однаковий macro F1, тож важливості параметрів можуть бути малоінформативними.

### Таблиця 3 — вкладена крос-валідація (варіант B, після оптимізації)

Зовнішній цикл: `StratifiedKFold(5)`; на кожному зовнішньому блоці `tune()` запускається **лише на його тренувальній частині**, а якість обчислюється на валідаційній. Оцінюється не конкретний набір гіперпараметрів, а *процедуру* «підбір + навчання».

In [ ]:
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
nested_scores = {}
for name in MODELS:
    t0, scores = time.time(), []
    for fold, (tr_idx, va_idx) in enumerate(outer.split(X_train, y_train), start=1):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        study_in = tune(name, X_tr, y_tr)
        model = build_pipeline(name, True, study_in.best_params).fit(X_tr, y_tr)
        scores.append(f1_score(y_va, model.predict(X_va), average="macro"))
    nested_scores[name] = np.array(scores)
    print(f"{MODEL_LABELS[name]:<14} nested macro F1 по блоках: {np.round(scores, 4)}  [{time.time()-t0:.0f} с]")

rows = []
for name in MODELS:
    ns = nested_scores[name]
    rows.append({"Модель": MODEL_LABELS[name],
                 "Nested CV macro F1 (сер. ± std)": f"{ns.mean():.4f} ± {ns.std(ddof=1):.4f}",
                 "study.best_value (звичайна оптимізація)": round(studies[name].best_value, 4),
                 "Δ = best_value − nested": round(studies[name].best_value - ns.mean(), 4)})
table3 = pd.DataFrame(rows).set_index("Модель")
display(table3)

✍️ **Коментар до таблиці 3.** `best_value` — максимум із багатьох зашумлених оцінок на тих самих блоках, за якими обирали гіперпараметри, тому він оптимістично зміщений (Cawley & Talbot, 2010). Вкладена CV оцінює якість на блоках, які не брали участі в підборі. Додатне Δ — міра цього оптимізму. На датасеті з ефектом стелі Δ може бути малим, а нестабільність рідкісних класів може навіть давати від'ємне Δ — це теж результат, який слід пояснити.

## 3.5 Статистичне порівняння — скоригований t-тест Наде–Бенжіо

$$t=\frac{\bar d}{\sqrt{\hat\sigma_d^2\left(\frac1{kr}+\frac{n_{test}}{n_{train}}\right)}},\qquad df=kr-1$$

Порівняння виконується на **однакових** блоках CV (п. 3.3). Якщо дисперсія різниць дорівнює нулю (обидві моделі дали однакові оцінки на всіх блоках — типово для ефекту стелі), тест вироджений: при $\bar d=0$ вважаємо $t=0$, $p=1$.

In [ ]:
def nadeau_bengio(scores1, scores2, n_train, n_test):
    d = np.asarray(scores1) - np.asarray(scores2)
    kr, d_bar, var = len(d), d.mean(), d.var(ddof=1)
    if var == 0:                                     # вироджений випадок (ефект стелі)
        return d_bar, (0.0 if d_bar == 0 else np.sign(d_bar) * np.inf), (1.0 if d_bar == 0 else 0.0)
    t_stat = d_bar / np.sqrt(var * (1 / kr + n_test / n_train))
    p_val = 2 * t_dist.sf(abs(t_stat), df=kr - 1)
    return d_bar, t_stat, p_val

_tr0, _te0 = next(iter(cv.split(X_train, y_train)))
n_tr, n_te = len(_tr0), len(_te0)
print(f"k·r = {len(raw_B['DT']['test_f1m'])}, n_train = {n_tr}, n_test = {n_te}")

def verdict(p): return "різниця значуща" if p < 0.05 else "різниця незначуща"

stat_rows = []
for name in MODELS:
    d, ts, p = nadeau_bengio(raw_A[name]["test_f1m"], raw_B[name]["test_f1m"], n_tr, n_te)
    stat_rows.append({"Порівняння": f"{MODEL_LABELS[name]}: A − B", "d̄": d, "t": ts, "p": p, "Висновок (α=0,05)": verdict(p)})

f1B = {n: raw_B[n]["test_f1m"].mean() for n in MODELS}
m1, m2 = sorted(MODELS, key=lambda n: f1B[n], reverse=True)[:2]
d, ts, p_best_two = nadeau_bengio(raw_B[m1]["test_f1m"], raw_B[m2]["test_f1m"], n_tr, n_te)
stat_rows.append({"Порівняння": f"{MODEL_LABELS[m1]} − {MODEL_LABELS[m2]} (варіант B)", "d̄": d, "t": ts, "p": p_best_two,
                  "Висновок (α=0,05)": verdict(p_best_two)})
table5 = pd.DataFrame(stat_rows).set_index("Порівняння")
display(table5.round(4))

✍️ **Коментар до таблиці 5.** Незначущий результат не означає «моделі однакові» — лише те, що за даними ефект не відрізняється від шуму. Навіть значуща різниця в соті частки macro F1 має обмежену практичну цінність.

## 3.6 Фінальна оцінка на тестовій вибірці

Моделі навчаються на всій тренувальній вибірці з найкращими параметрами Optuna (варіант B) і зберігаються. Тестова вибірка використовується **лише тут** (та для інтерпретації в п. 3.7–3.8).

«Найкраща» модель обирається **за вкладеною CV** (середній macro F1; при рівності — менше std), а не за тестом, щоб не обирати модель за тестовою вибіркою.

In [ ]:
final_models, test_rows = {}, []
for name in MODELS:
    pipe = build_pipeline(name, True, studies[name].best_params).fit(X_train, y_train)
    final_models[name] = pipe
    joblib.dump(pipe, f"models/{name}_final.joblib")
    pred = pipe.predict(X_test)
    test_rows.append({"Модель": MODEL_LABELS[name],
                      "Accuracy": accuracy_score(y_test, pred),
                      "Macro F1": f1_score(y_test, pred, average="macro"),
                      "Balanced acc.": balanced_accuracy_score(y_test, pred)})
table6 = pd.DataFrame(test_rows).set_index("Модель")
print("Таблиця 6 — якість фінальних моделей на тесті:")
display(table6.round(4))

best_name = max(MODELS, key=lambda n: (nested_scores[n].mean(), -nested_scores[n].std(ddof=1)))
best_model = final_models[best_name]
best_pred = best_model.predict(X_test)
print("Найкраща модель за nested CV:", MODEL_LABELS[best_name])
labels_all = np.arange(len(class_names))
print(classification_report(y_test, best_pred, labels=labels_all, target_names=class_names, zero_division=0))
report = classification_report(y_test, best_pred, labels=labels_all, target_names=class_names,
                               zero_division=0, output_dict=True)

✍️ **Чому тест відрізняється від CV.** Тест містить лише ≈ 20 % даних, а найменші класи представлені 1–2 об'єктами: одна помилка змінює recall такого класу на 50–100 %, а macro F1 — на десятки відсотків. Тому тестові значення значно шумніші за CV-оцінки (які усереднюють 15 блоків).

### Графік 5 — матриці плутанини найкращої моделі

In [ ]:
cm = confusion_matrix(y_test, best_pred, labels=labels_all)
cm_norm = confusion_matrix(y_test, best_pred, labels=labels_all, normalize="true")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title(f"Матриця плутанини — {MODEL_LABELS[best_name]}")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Нормалізована по рядках (recall)")
for ax in axes: ax.set_xlabel("Передбачений клас"); ax.set_ylabel("Справжній клас")
plt.tight_layout(); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(class_names[i], class_names[j], int(off[i, j])) for i in range(len(cm)) for j in range(len(cm)) if off[i, j] > 0],
               key=lambda x: -x[2])
if pairs:
    print("Найчастіше плутаються (справжній → передбачений: кількість):")
    for a, b, c in pairs: print(f"  {a} → {b}: {c}")
else:
    print("На тесті помилок немає — класи не плутаються (ефект стелі).")

## 3.7 Інтерпретація: важливість ознак (MDI та permutation importance)

* **MDI** (`feature_importances_`) обчислюється на *оброблених* ознаках (після one-hot і з індикаторами), назви — через `get_feature_names_out()`.
* **Permutation importance** — на тестовій вибірці, застосована до **всього пайплайну** (тобто на початкових ознаках), `scoring='f1_macro'`, 10 повторень.
* Для порівняння MDI і permutation importance MDI додатково агрегується до початкових ознак (сума по one-hot-колонках і індикатору).

In [ ]:
def to_original(feat, columns):
    if feat.startswith("miss__missingindicator_"):
        return feat[len("miss__missingindicator_"):]
    if feat.startswith("num__"):
        return feat[len("num__"):]
    rest = feat[len("cat__"):]
    cands = [c for c in columns if rest.startswith(c + "_") or rest == c]
    return max(cands, key=len) if cands else rest

imp_store, summary_rows = {}, []
for name, pipe in final_models.items():
    prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
    fnames = prep.get_feature_names_out()
    mdi = pd.Series(clf.feature_importances_, index=fnames)
    top_mdi = mdi.sort_values(ascending=False).head(15)

    perm = permutation_importance(pipe, X_test, y_test, scoring="f1_macro", n_repeats=10, random_state=SEED, n_jobs=1)
    perm_mean = pd.Series(perm.importances_mean, index=X_test.columns)
    perm_std = pd.Series(perm.importances_std, index=X_test.columns)
    top_perm = perm_mean.sort_values(ascending=False).head(15)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    colors = ["tab:red" if "missingindicator" in n else "tab:blue" for n in top_mdi.index[::-1]]
    axes[0].barh(top_mdi.index[::-1], top_mdi.values[::-1], color=colors)
    axes[0].set_title(f"{MODEL_LABELS[name]}: top-15 MDI (червоні — індикатори пропусків)")
    axes[0].set_xlabel("Важливість (MDI)")
    axes[1].barh(top_perm.index[::-1], top_perm.values[::-1], xerr=perm_std[top_perm.index[::-1]].values, color="tab:green")
    axes[1].set_title(f"{MODEL_LABELS[name]}: top-15 permutation importance (тест)")
    axes[1].set_xlabel("Спад macro F1 після перестановки")
    plt.tight_layout(); plt.show()

    n_ind = int(sum("missingindicator" in n for n in top_mdi.index))
    mdi_orig = mdi.groupby([to_original(n, list(X_train.columns)) for n in mdi.index]).sum()
    common = mdi_orig.index.intersection(perm_mean.index)
    rho = spearmanr(mdi_orig[common], perm_mean[common])[0] if common.size > 2 else np.nan
    overlap = len(set(mdi_orig.sort_values(ascending=False).head(15).index) & set(top_perm.index))
    imp_store[name] = {"mdi_top": top_mdi, "perm_top": top_perm, "n_ind": n_ind}
    summary_rows.append({"Модель": MODEL_LABELS[name], "Індикаторів у top-15 MDI": n_ind,
                         "Spearman ρ (MDI vs perm, початкові ознаки)": round(rho, 3) if rho == rho else np.nan,
                         "Збіг top-15 (з 15)": overlap,
                         "Ознак з perm > 0": int((perm_mean > 0).sum())})
    print(f"{MODEL_LABELS[name]}: індикатори в top-15 MDI → {[n for n in top_mdi.index if 'missingindicator' in n]}\n")

table_imp = pd.DataFrame(summary_rows).set_index("Модель")
display(table_imp)

✍️ **Відповіді на запитання п. 3.7.**

* *Індикатори в top-15 MDI.* Якщо `missingindicator_*` потрапили в top-15 — пропуск несе інформацію про клас, що узгоджується з χ²-аналізом п. 3.1 (пропуски не MCAR, частина структурні). Якщо ні — або класи й так добре розділяються іншими ознаками (ефект стелі), або індикатор дублює ознаку, що вже є в моделі.
* *Узгодженість MDI та permutation.* Дивіться ρ Спірмена та збіг top-15 у таблиці. MDI рахується на тренувальних даних і зміщена на користь ознак із багатьма унікальними значеннями / безперервних (більше можливих порогів розщеплення; Strobl et al., 2007), тоді як permutation importance вимірюється на тесті й показує реальний спад якості; для корельованих ознак перестановка однієї може не зменшувати якість (її замінює інша), тому permutation їх недооцінює, а MDI розподіляє важливість між ними довільно.
* *Чому важливості різняться між DT, RF, XGB.* Одиночне дерево жорстко використовує 1–2 «найкращі» ознаки; RF внаслідок випадкового підпростору та усереднення розподіляє важливість між корельованими ознаками; XGB послідовно виправляє залишки й використовує інші ознаки для «важких» об'єктів.

Запишіть власні висновки за вашими графіками.

## 3.8 Аналіз розбіжностей між моделями

Об'єкт розбіжності — об'єкт, на якому прогнози трьох фінальних моделей не всі однакові. Якщо в тесті їх менше трьох, таблицю доповнюємо об'єктами з тренувальної вибірки з **out-of-fold** прогнозами (`cross_val_predict`; прогнози моделей на їхніх власних тренувальних даних не використовуються) і явно позначаємо це в стовпчику «Джерело».

In [ ]:
def pred_names(arr): return [class_names[int(v)] for v in arr]

test_preds = {n: final_models[n].predict(X_test) for n in MODELS}
test_df = pd.DataFrame({"Справжній клас": pred_names(y_test.values),
                        "Decision Tree": pred_names(test_preds["DT"]),
                        "Random Forest": pred_names(test_preds["RF"]),
                        "XGBoost": pred_names(test_preds["XGB"]),
                        "Джерело": "тест"}, index=X_test.index)
pc = ["Decision Tree", "Random Forest", "XGBoost"]
dis_test = test_df[test_df[pc].nunique(axis=1) > 1]
print(f"Об'єктів розбіжності в тесті: {len(dis_test)} із {len(test_df)}")

parts = [dis_test]
if len(dis_test) < 3:
    print("→ Менше трьох: додаємо об'єкти з train за out-of-fold прогнозами (позначено «train (OOF)»).")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = {n: cross_val_predict(build_pipeline(n, True, studies[n].best_params), X_train, y_train, cv=skf, n_jobs=1)
           for n in MODELS}
    oof_df = pd.DataFrame({"Справжній клас": pred_names(y_train.values),
                           "Decision Tree": pred_names(oof["DT"]), "Random Forest": pred_names(oof["RF"]),
                           "XGBoost": pred_names(oof["XGB"]), "Джерело": "train (OOF)"}, index=X_train.index)
    dis_oof = oof_df[oof_df[pc].nunique(axis=1) > 1]
    print(f"Об'єктів розбіжності в train (OOF): {len(dis_oof)} із {len(oof_df)}")
    parts.append(dis_oof.head(max(3 - len(dis_test), 3)))

table7 = pd.concat(parts)
table7.index.name = "Об'єкт"
print("\nТаблиця 7 — об'єкти розбіжностей:")
display(table7)
if len(table7) < 3:
    print("⚠ Знайдено менше трьох об'єктів розбіжності — зафіксуйте це у звіті.")

### Детальний аналіз одного об'єкта розбіжності

1. початкові значення всіх ознак, позначки пропусків і значення, якими їх заповнено імпутером;
2. SHAP-значення (`shap.TreeExplainer`) для кожної моделі — топ-5 ознак за модулем внеску в **прогнозований моделлю клас**;
3. порівняння моделей та `predict_proba`.

In [ ]:
idx = table7.index[0]
src = table7.loc[idx, "Джерело"]
print(f"Аналізований об'єкт: #{idx} (джерело: {src}); справжній клас = {table7.loc[idx, 'Справжній клас']}")

prep_any = final_models["XGB"].named_steps["prep"]
cols_by = {n: c for n, _, c in prep_any.transformers_}
fill = dict(zip(cols_by["num"], prep_any.named_transformers_["num"].statistics_))
fill.update(dict(zip(cols_by["cat"], prep_any.named_transformers_["cat"].named_steps["impute"].statistics_)))

row = X.loc[idx]
detail = pd.DataFrame({
    "Початкове значення": [("NaN" if pd.isna(row[c]) else row[c]) for c in X.columns],
    "Пропущено": ["так" if pd.isna(row[c]) else "" for c in X.columns],
    "Заповнено імпутером": [(fill.get(c) if pd.isna(row[c]) else "") for c in X.columns],
}, index=X.columns)
detail.index.name = "Ознака"
display(detail)
print("\nПропущені ознаки цього об'єкта (і чим їх заповнено):")
display(detail[detail["Пропущено"] == "так"])

### Графік 6 — SHAP-внески для об'єкта розбіжності

In [ ]:
x_row = X.loc[[idx]]
missing_cols = set(X.columns[x_row.iloc[0].isna().values])
shap_out, proba_tbl, add_checks = {}, {}, []

for name in MODELS:
    pipe = final_models[name]
    prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
    xp = prep.transform(x_row)
    fnames = prep.get_feature_names_out()
    expl = shap.TreeExplainer(clf)
    sv = expl.shap_values(xp, check_additivity=False)
    sv = np.stack(sv, axis=-1)[0] if isinstance(sv, list) else np.asarray(sv)[0]   # (n_features, n_classes)
    base = np.atleast_1d(expl.expected_value)
    pred_c = int(pipe.predict(x_row)[0])
    proba = pipe.predict_proba(x_row)[0]
    proba_tbl[MODEL_LABELS[name]] = proba

    contrib = pd.Series(sv[:, pred_c], index=fnames)
    top5 = contrib.reindex(contrib.abs().sort_values(ascending=False).head(5).index)
    shap_out[name] = (pred_c, top5)

    # перевірка адитивності: base + ΣSHAP = вихід моделі в просторі, в якому рахує TreeExplainer
    if name == "XGB":
        target = clf.predict(xp, output_margin=True)[0][pred_c]; space = "log-odds (margin)"
    else:
        target = proba[pred_c]; space = "ймовірність"
    add_checks.append((MODEL_LABELS[name], base[pred_c] + sv[:, pred_c].sum(), target, space))

print("Адитивність SHAP (E[f] + ΣSHAP ≈ вихід моделі для прогнозованого класу):")
for n, lhs, rhs, sp in add_checks:
    print(f"  {n:<14} {lhs:.4f} ≈ {rhs:.4f}   [{sp}]")

print("\nТоп-5 ознак за |SHAP| у прогнозований клас:")
for name in MODELS:
    pred_c, top5 = shap_out[name]
    tbl = pd.DataFrame({"SHAP-внесок": top5.round(4),
                        "Початкова ознака": [to_original(n, list(X.columns)) for n in top5.index]})
    tbl["Тип"] = ["індикатор пропуску" if "missingindicator" in n else
                  ("значення було імпутоване" if to_original(n, list(X.columns)) in missing_cols else "спостережене значення")
                  for n in top5.index]
    print(f"\n{MODEL_LABELS[name]} → прогноз: клас {class_names[pred_c]}")
    display(tbl)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, name in zip(axes, MODELS):
    pred_c, top5 = shap_out[name]
    ax.barh(top5.index[::-1], top5.values[::-1], color=["tab:red" if v > 0 else "tab:blue" for v in top5.values[::-1]])
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"{MODEL_LABELS[name]} → клас {class_names[pred_c]}")
    ax.set_xlabel("SHAP-внесок")
plt.suptitle(f"SHAP: топ-5 ознак для об'єкта #{idx}", y=1.03)
plt.tight_layout(); plt.show()

print("predict_proba (рядки — класи, стовпчики — моделі):")
display(pd.DataFrame(proba_tbl, index=class_names).round(4))
for name in MODELS:
    p = np.sort(final_models[name].predict_proba(x_row)[0])[::-1]
    print(f"{MODEL_LABELS[name]:<14} p(top-1) = {p[0]:.3f}, p(top-2) = {p[1]:.3f}, відрив = {p[0]-p[1]:.3f}")

✍️ **Пояснення розбіжності** (за вашими числами вище): (1) чи були серед найвагоміших ознак імпутовані значення або індикатори пропусків — медіана/мода «вигадує» значення, яке різні моделі інтерпретують по-різному; (2) одиночне дерево приймає рішення за кількома жорсткими порогами, ансамблі усереднюють багато дерев (RF) або сумують поправки (XGB), тому реагують на ті самі ознаки м'якше; (3) відрив між `p(top-1)` і `p(top-2)` показує, чи лежить об'єкт біля межі між класами — у цьому випадку навіть невеликі відмінності моделей дають різні мітки.

## Висновок

Нижче автоматично зібрано числові факти з ваших результатів — на їх основі напишіть висновок (8–12 речень) за 7 питаннями методички.

In [ ]:
nm = {n: nested_scores[n].mean() for n in MODELS}
ns = {n: nested_scores[n].std(ddof=1) for n in MODELS}
fA = {n: raw_A[n]["test_f1m"].mean() for n in MODELS}
fB = {n: raw_B[n]["test_f1m"].mean() for n in MODELS}

print("1) Найкраща модель за nested macro F1:", MODEL_LABELS[best_name],
      f"({nm[best_name]:.4f} ± {ns[best_name]:.4f}); різниця між двома найкращими у варіанті B "
      f"({MODEL_LABELS[m1]} vs {MODEL_LABELS[m2]}): p = {p_best_two:.4f} → {verdict(p_best_two)}.")

print("\n2) Індикатори пропусків (variant B − A, CV macro F1) та p-значення Наде–Бенжіо:")
for n in MODELS:
    p = table5.loc[f"{MODEL_LABELS[n]}: A − B", "p"]
    print(f"   {MODEL_LABELS[n]:<14} A={fA[n]:.4f}, B={fB[n]:.4f}, B−A={fB[n]-fA[n]:+.4f}, p={p:.4f}; "
          f"індикаторів у top-15 MDI: {imp_store[n]['n_ind']}")
print(f"   Ознак із пропусками, пов'язаних із класом за χ²: {n_dep} із {len(chi_df)}.")

print("\n3) best_value vs nested CV:")
for n in MODELS:
    print(f"   {MODEL_LABELS[n]:<14} best_value={studies[n].best_value:.4f}, nested={nm[n]:.4f}, Δ={studies[n].best_value-nm[n]:+.4f}")

print("\n4) Зміна якості після оптимізації (nested CV − типові параметри, варіант B, CV):")
gain = {n: nm[n] - fB[n] for n in MODELS}
for n in MODELS: print(f"   {MODEL_LABELS[n]:<14} {gain[n]:+.4f}")
print("   Найбільший приріст:", MODEL_LABELS[max(gain, key=gain.get)])

print("\n5) Найскладніші класи на тесті (F1 найкращої моделі, підтримка):")
cls_f1 = sorted([(c, report[c]["f1-score"], int(report[c]["support"])) for c in class_names], key=lambda x: x[1])
for c, f, s in cls_f1[:3]: print(f"   клас {c}: F1={f:.3f}, support={s}, у train={int((y_train==class_names.index(c)).sum())}")

print(f"\n6) Об'єктів розбіжності: у тесті {len(dis_test)}; у таблиці 7 всього {len(table7)}.")

print("\n7) Якість / стабільність / вартість:")
for n in MODELS:
    print(f"   {MODEL_LABELS[n]:<14} nested F1={nm[n]:.4f} ± {ns[n]:.4f}; fit у CV ≈ {raw_B[n]['fit_time'].mean():.2f} с; "
          f"час tune() ≈ {tune_time[n]:.0f} с")

✍️ **Ваш висновок (8–12 речень).** Відповідайте на 7 питань методички, спираючись на числа вище. Пам'ятайте: «правильної» моделі наперед не існує; якщо різниці статистично незначущі — це теж результат, який потрібно проаналізувати (ефект стелі, мала кількість об'єктів рідкісних класів).

## Контрольні питання (стислі відповіді)

1. **RF vs дерево.** Одне глибоке дерево має малий зсув, але велику дисперсію. RF усереднює $B$ дерев: дисперсія середнього ≈ $\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$, зсув майже не змінюється. Бутстреп і випадковий підпростір ознак (`max_features`) **декорелюють** дерева (зменшують $\rho$), а саме $\rho$ обмежує виграш від збільшення $B$.
2. **Бустинг vs беггінг.** Беггінг будує незалежні дерева паралельно й зменшує дисперсію; бустинг — послідовно, кожне нове дерево виправляє помилки (градієнт втрат) попередніх, зменшуючи переважно зсув. Бустинг сильніше схильний до перенавчання зі зростанням числа дерев (без `learning_rate`, регуляризації, early stopping); RF із додаванням дерев не перенавчається (дисперсія збігається до межі).
3. **Macro F1 / balanced accuracy.** За сильного дисбалансу accuracy визначається домінантним класом (константний прогноз уже дає високе значення). Macro F1 — просте середнє F1 по класах (кожен клас має однакову вагу), balanced accuracy — середній recall по класах. Weighted F1 зважує класи за підтримкою, тому майже повторює accuracy.
4. **MCAR/MAR/MNAR.** MCAR — пропуск не залежить ні від чого; MAR — залежить від спостережених ознак; MNAR — від самого пропущеного значення/неспостережених чинників. Індикатори найкорисніші, коли пропуск інформативний (не MCAR, особливо MNAR/структурні пропуски). Лише за даними MAR і MNAR однозначно розрізнити **не можна** — потрібні знання про процес збору даних або аналіз чутливості.
5. **Витік даних.** `fit` імпутера/кодувальника на всьому датасеті до поділу чи CV; визначення індикаторів за всіма даними; передача в Optuna заздалегідь оброблених даних; використання тесту в цільовій функції чи для вибору моделі. Запобігання: усе у `Pipeline`, підбір лише на train, тест — один раз.
6. **best_value та nested CV.** `best_value` — максимум із багатьох зашумлених оцінок на тих самих блоках, за якими вибирали параметри («прокляття переможця»), тому зміщений догори. У nested CV зовнішній цикл оцінює якість на блоках, що не брали участі в підборі; на кожному зовнішньому блоці параметри підбираються заново, отже оцінюється **процедура** «підбір + навчання», а не одна конкретна модель.
7. **Парний t-тест і Наде–Бенжіо.** Блоки CV мають спільні тренувальні дані, тож різниці $d_j$ залежні, і звичайний тест занижує дисперсію (забагато хибно позитивних). Поправка замінює множник $1/(kr)$ на $1/(kr)+n_{test}/n_{train}$.
8. **MDI vs permutation.** MDI — сумарне зменшення неоднорідності на тренувальних даних (зміщене на користь ознак із багатьма унікальними значеннями); permutation — спад метрики на відкладених даних після перемішування ознаки (модельно-незалежна). Для корельованих ознак MDI ділить важливість між ними довільно, permutation — недооцінює обидві (одна компенсує іншу) і може створювати нереалістичні об'єкти.
9. **SHAP.** Значення Шеплі з теорії кооперативних ігор: внесок ознаки в різницю між прогнозом і середнім передбаченням. Адитивність: $f(x)=\phi_0+\sum_i\phi_i$. Локальне пояснення стосується одного об'єкта; глобальна важливість — це, наприклад, середнє $|\phi_i|$ по набору.
10. **XGBoost і пропуски.** Sparsity-aware split finding: для кожного розщеплення пропущені значення спрямовуються спершу ліворуч, потім праворуч, обирається напрямок із більшим приростом якості (default direction), який запам'ятовується й використовується при прогнозі; перебираються лише непропущені значення.
11. **Optuna.** TPE моделює «добрі» та «погані» області простору й семплює перспективні точки (ефективніше за повний перебір GridSearchCV і випадковий RandomizedSearchCV); прунінг достроково зупиняє непромовісні спроби; гнучкі (умовні, логарифмічні) простори пошуку, візуалізація й паралелізація.